# CML6032 ASSIGNMENT 1 - SLIDE VIDEO FROM RECORDED VOICE NOTES

NISHEAL MICHAEL KALEY | ENTRY NO 2025CYS7090 | DEPARTMENT OF CHEMISTRY, IIT DELHI

---

## WHAT CHANGED

THE EARLIER NOTEBOOK SYNTHESISED THE NARRATION WITH A TEXT-TO-SPEECH MODEL AND
NEEDED A T4 GPU FOR 15 TO 25 MINUTES. THAT IS RETIRED. THE NARRATION IS NOW THE
AUTHOR'S OWN RECORDED VOICE NOTES, ONE PER NUMBERED SLIDE, SO THERE IS NO MODEL,
NO GPU, NO MODEL DOWNLOAD AND NO REFERENCE AUDIO. THE WORK IS AUDIO CONDITIONING
AND ONE ENCODE PASS.

| | OLD TTS NOTEBOOK | THIS NOTEBOOK |
|---|---|---|
| VOICE | SYNTHESISED FROM TRANSCRIPT.PY | THE AUTHOR'S OWN RECORDINGS |
| RUNTIME TYPE | GPU, T4 | **CPU IS ENOUGH**, SELECT NONE |
| WALL CLOCK | 15 TO 25 MINUTES | 4 TO 8 MINUTES, MOSTLY THE ENCODE |
| NETWORK | MODEL WEIGHTS DOWNLOAD | NONE AFTER THE FILES ARE IN PLACE |
| FAILURE MODE TO FEAR | WRONG OR DRIFTING SYNTHETIC SPEECH | A MISSING OR TRUNCATED TAKE |

BECAUSE NO GPU IS NEEDED, THIS WILL ALSO RUN UNCHANGED ON A LOCAL WINDOWS MACHINE
WITH PYTHON AND FFMPEG ON PATH: `python BUILD_VIDEO.py`. THAT IS USUALLY FASTER
THAN UPLOADING ABOUT 130 MB OF WAV TO COLAB. THE NOTEBOOK EXISTS FOR THE CASE
WHERE THE LOCAL MACHINE IS NOT CONVENIENT.

## WHAT YOU NEED IN PLACE

```
WORKDIR/
  slides/        18 PNG: 01..15 ANSWER SLIDES, PLUS 01a, 04a, 09a QUESTION SLIDES
  voicenotes/    15 RECORDINGS, ONE PER ANSWER SLIDE
```

RECORDING FILENAMES ARE MATCHED LOOSELY, SO ALL OF THESE WORK:
`Slide 7.wav`, `slide_7.wav`, `07.wav`, `7.m4a`, `Slide07.mp3`.
CONTAINERS ACCEPTED: wav m4a mp3 aac flac ogg opus webm.

THE THREE QUESTION SLIDES ARE NOT NARRATED. THEY ARE HELD ON SCREEN IN SILENCE
FOR `CML_QUESTION_HOLD` SECONDS, DEFAULT 8. RAISE IT IF YOU WANT LONGER READING
TIME, OR RECORD NOTES FOR THEM AND NAME THEM `Slide 1a.wav` STYLE ONLY AFTER
EXTENDING THE SCRIPT; AS SHIPPED THEY ARE DELIBERATELY SILENT.


## 1. ENVIRONMENT

RUNTIME TYPE SHOULD BE **CPU**. A GPU IS NOT WRONG, IT IS JUST WASTED; IF ONE IS
PRESENT THE SCRIPT WILL USE `h264_nvenc` AND FINISH THE ENCODE SOONER.


In [ ]:
import shutil, subprocess, sys, os
print(sys.version.split()[0])
if not shutil.which('ffmpeg'):
    print('INSTALLING FFMPEG')
    subprocess.run('apt-get -qq update && apt-get -qq install -y ffmpeg', shell=True, check=True)
print(subprocess.run(['ffmpeg','-version'], capture_output=True, text=True).stdout.splitlines()[0])
print('NVENC', 'YES' if 'h264_nvenc' in subprocess.run(['ffmpeg','-hide_banner','-encoders'],
      capture_output=True, text=True).stdout else 'NO, LIBX264 WILL BE USED')


## 2. WORKING DIRECTORY AND INPUTS

PICK ONE OF THE TWO CELLS BELOW. GOOGLE DRIVE IS STRONGLY PREFERRED BECAUSE THE
RECORDINGS ARE LARGE AND A DISCONNECT WOULD OTHERWISE COST YOU THE UPLOAD.


In [ ]:
# ---- OPTION A, GOOGLE DRIVE (RECOMMENDED) ----------------------------------
# PUT slides/ AND voicenotes/ INSIDE A FOLDER ON YOUR DRIVE, THEN SET WORKDIR.
from google.colab import drive
drive.mount('/content/drive')
WORKDIR = '/content/drive/MyDrive/CML6032_video'   # EDIT THIS
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print(os.getcwd())


In [ ]:
# ---- OPTION B, DIRECT UPLOAD ------------------------------------------------
# UPLOAD ONE ZIP THAT CONTAINS slides/ AND voicenotes/ AT ITS TOP LEVEL.
# WORK IS LOST IF THE SESSION DROPS, SO PREFER OPTION A.
# from google.colab import files
# import zipfile
# WORKDIR = '/content/CML6032_video'
# os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
# up = files.upload()
# for name in up:
#     if name.lower().endswith('.zip'):
#         zipfile.ZipFile(name).extractall('.')
# print(sorted(os.listdir('.')))


## 3. WRITE THE BUILDER

THE CELL BELOW WRITES `BUILD_VIDEO.py` INTO THE WORKING DIRECTORY SO THIS
NOTEBOOK IS SELF CONTAINED AND NEEDS NOTHING FETCHED FROM THE REPOSITORY. IT IS
BYTE FOR BYTE THE SAME SCRIPT THAT SHIPS BESIDE THE NOTEBOOK; IF YOU ALREADY
UPLOADED THAT FILE YOU CAN SKIP THIS CELL.


In [ ]:
%%writefile BUILD_VIDEO.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CML6032 ASSIGNMENT 1 - SLIDE VIDEO ASSEMBLY FROM RECORDED VOICE NOTES
NISHEAL MICHAEL KALEY | ENTRY NO 2025CYS7090 | IIT DELHI

WHAT THIS REPLACES
------------------
THE EARLIER TRANSCRIPT.PY SYNTHESISED NARRATION WITH A TEXT-TO-SPEECH MODEL AND
NEEDED A GPU. THAT IS RETIRED. THE NARRATION IS NOW THE AUTHOR'S OWN RECORDED
VOICE NOTES, ONE PER NUMBERED SLIDE, SO THIS SCRIPT ONLY CONDITIONS AUDIO AND
MUXES. IT IS CPU ONLY, NEEDS NO MODEL DOWNLOAD AND NEEDS NO NETWORK.

INPUTS
------
  CML_SLIDES   DIRECTORY OF SLIDE PNG. DEFAULT ./slides
               FILES NAMED 01.png .. 15.png ARE ANSWER SLIDES AND ARE NARRATED.
               FILES WITH A LETTER SUFFIX (01a.png, 04a.png, 09a.png) ARE
               QUESTION SLIDES: THEY ARE NOT NARRATED AND ARE HELD SILENT.
  CML_VOICE    DIRECTORY OF RECORDINGS. DEFAULT ./voicenotes
               ONE FILE PER ANSWER SLIDE. ACCEPTED NAMES, CASE AND SPACING FREE:
               "Slide 7.wav", "slide_7.wav", "07.wav", "7.m4a", "Slide07.mp3" ...
               ACCEPTED CONTAINERS: wav m4a mp3 aac flac ogg opus webm
  CML_WORK     SCRATCH AND CACHE DIRECTORY. DEFAULT ./build
  CML_OUT      OUTPUT MP4. DEFAULT ./CML6032_Assignment1_Nisheal_2025CYS7090.mp4

OUTPUTS
-------
  THE MP4, 1920 x 1080, 30 fps, H.264 + AAC, FASTSTART
  chapters.json   ONE ENTRY PER SLIDE FROM THE REAL MEASURED DURATIONS
  timeline.csv    PER SLIDE START, DURATION AND SOURCE FILE
  build/norm/     PER SLIDE CONDITIONED WAV, REUSED ON RERUNS

AUDIO CONDITIONING PER SLIDE
----------------------------
  1 HIGHPASS AT 70 Hz TO DROP DESK RUMBLE AND BREATH THUMP
  2 OPTIONAL FFT DENOISE, OFF BY DEFAULT, SET CML_DENOISE=1 TO ENABLE
  3 LEADING AND TRAILING SILENCE TRIMMED AT -50 dB PEAK
  4 TWO PASS EBU R128 LOUDNESS NORMALISATION TO I = -16 LUFS, TP = -1.5 dBTP
    TWO PASS MATTERS: A SINGLE PASS IS DYNAMIC AND PUMPS THE QUIET SLIDES
  5 RESAMPLED TO 48 kHz STEREO s16 SO EVERY SEGMENT CONCATENATES CLEANLY
  6 0.25 s OF LEAD IN AND 0.60 s OF LEAD OUT SO CUTS DO NOT CLIP SPEECH

SAFETY CHECKS, ALL FATAL UNLESS STATED
--------------------------------------
  * EVERY ANSWER SLIDE MUST HAVE EXACTLY ONE RECORDING
  * A RECORDING THAT MATCHES NO SLIDE IS AN ERROR, NOT A SILENT SKIP
  * TWO RECORDINGS CLAIMING THE SAME SLIDE IS AN ERROR
  * A RECORDING SHORTER THAN 0.35 OF THE MEDIAN IS REPORTED AS A LIKELY
    TRUNCATED TAKE; IT IS A WARNING, AND CML_STRICT=1 PROMOTES IT TO FATAL
  * THE FINISHED MP4 DURATION IS COMPARED WITH THE PLANNED TIMELINE AND MUST
    AGREE TO WITHIN 0.5 s

USAGE
-----
  python3 BUILD_VIDEO.py            BUILD
  python3 BUILD_VIDEO.py --check    VALIDATE INPUTS AND PRINT THE PLAN ONLY
  python3 BUILD_VIDEO.py --force    IGNORE THE CACHE AND RECONDITION EVERY NOTE
  python3 BUILD_VIDEO.py --meta     REWRITE chapters.json AND timeline.csv ONLY,
                                    FROM THE CACHED AUDIO, WITHOUT RE-ENCODING
"""

import csv, json, os, re, shutil, subprocess, sys
from pathlib import Path

# ----------------------------------------------------------------- CONFIG
HERE   = Path(__file__).resolve().parent
SLIDES = Path(os.environ.get('CML_SLIDES', HERE / 'slides'))
VOICE  = Path(os.environ.get('CML_VOICE',  HERE / 'voicenotes'))
WORK   = Path(os.environ.get('CML_WORK',   HERE / 'build'))
OUT    = Path(os.environ.get('CML_OUT',    HERE / 'CML6032_Assignment1_Nisheal_2025CYS7090.mp4'))

FPS           = int(os.environ.get('CML_FPS', 30))
CRF           = int(os.environ.get('CML_CRF', 20))
ABR           = os.environ.get('CML_ABR', '160k')
TARGET_I      = -16.0
TARGET_TP     = -1.5
TARGET_LRA    = 11.0
HIGHPASS_HZ   = 70
DENOISE       = os.environ.get('CML_DENOISE', '0') == '1'
TRIM_DB       = -50
HEAD_PAD      = float(os.environ.get('CML_HEAD_PAD', 0.25))
TAIL_PAD      = float(os.environ.get('CML_TAIL_PAD', 0.60))
QUESTION_HOLD = float(os.environ.get('CML_QUESTION_HOLD', 8.0))
SHORT_FACTOR  = 0.35
STRICT        = os.environ.get('CML_STRICT', '0') == '1'

AUDIO_EXT = ('.wav', '.m4a', '.mp3', '.aac', '.flac', '.ogg', '.opus', '.webm')

TITLES = {
     1: "TITLE AND SCOPE",
     2: "CRYSTAL MORPHOLOGY AND MOST PROBABLE POINT GROUPS",
     3: "SPECIMEN 1: CUBO-OCTAHEDRAL HABIT",
     4: "SPECIMEN 2: HEXAGONAL PRISMATIC HABIT",
     5: "PART B SEGMENT 1: FERROELECTRIC DISTORTION OF BaTiO3",
     6: "Ti DISPLACEMENT SERIES ALONG c",
     7: "NINE DISTORTION STATES: GEOMETRY AND RIGID-ION POLARISATION",
     8: "FERROELECTRIC HYSTERESIS: CORRECTED TWO-BRANCH LOOP",
     9: "WHY A LOOP AND NOT A LINE: DOUBLE WELL, DOMAINS AND Tc",
    10: "PART B SEGMENT 2: NINE BINARY CUBIC PROTOTYPES",
    11: "ROCK SALT TYPE: NaCl AND CaTe",
    12: "CAESIUM CHLORIDE TYPE: CsCl AND AuZn",
    13: "FLUORITE AND ANTIFLUORITE: CaF2, CeO2 AND K2O",
    14: "ZINC BLENDE TYPE: ZnS AND GaP",
    15: "VERIFICATION, MANIFEST AND WHAT IS STILL MISSING",
}
QUESTION_TITLES = {
    '01a': "THE BRIEF: PART A AS ISSUED",
    '04a': "THE BRIEF: PART B SEGMENT 1 AS ISSUED",
    '09a': "THE BRIEF: PART B SEGMENT 2 AS ISSUED",
}

# ------------------------------------------------------------------ UTIL
class Fail(SystemExit):
    def __init__(self, msg): super().__init__('FATAL: ' + msg)

def sh(args, capture=True):
    r = subprocess.run(args, stdout=subprocess.PIPE if capture else None,
                       stderr=subprocess.STDOUT if capture else None, text=True)
    return r.returncode, (r.stdout or '')

def need(tool):
    if not shutil.which(tool): raise Fail(f'{tool} IS NOT ON PATH')

def duration(path):
    rc, out = sh(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                  '-of', 'default=nw=1:nk=1', str(path)])
    if rc or not out.strip(): raise Fail(f'CANNOT PROBE {path}')
    return float(out.strip())

def hms(t):
    t = int(round(t)); return f'{t//60:02d}:{t%60:02d}'

# ------------------------------------------------- SLIDE AND NOTE PAIRING
SLIDE_RE = re.compile(r'^(\d{1,2})([a-z]?)$', re.I)

def read_slides():
    if not SLIDES.is_dir(): raise Fail(f'NO SLIDE DIRECTORY AT {SLIDES}')
    out = []
    for p in sorted(SLIDES.glob('*.png')):
        m = SLIDE_RE.match(p.stem)
        if not m:
            print(f'  SKIPPED NON SLIDE FILE {p.name}'); continue
        out.append({'path': p, 'key': p.stem.lower(),
                    'n': int(m.group(1)), 'suffix': m.group(2).lower()})
    if not out: raise Fail(f'NO SLIDE PNG FOUND IN {SLIDES}')
    out.sort(key=lambda s: (s['n'], s['suffix']))
    return out

NOTE_RE = re.compile(r'(?:^|[^0-9])(\d{1,2})\s*$')

def note_index(stem):
    s = re.sub(r'[\s_\-]+', ' ', stem.strip())
    s = re.sub(r'(?i)^(slide|sl|s|track|rec|recording|voice ?note|note)\b', '', s).strip()
    m = re.search(r'(\d{1,2})', s)
    return int(m.group(1)) if m else None

def read_notes():
    if not VOICE.is_dir(): raise Fail(f'NO VOICE NOTE DIRECTORY AT {VOICE}')
    found, unmatched = {}, []
    for p in sorted(VOICE.iterdir()):
        if not p.is_file() or p.suffix.lower() not in AUDIO_EXT: continue
        i = note_index(p.stem)
        if i is None: unmatched.append(p); continue
        if i in found:
            raise Fail(f'TWO RECORDINGS CLAIM SLIDE {i}: '
                       f'{found[i].name} AND {p.name}')
        found[i] = p
    if unmatched:
        raise Fail('THESE RECORDINGS CARRY NO SLIDE NUMBER: '
                   + ', '.join(p.name for p in unmatched))
    if not found: raise Fail(f'NO RECORDINGS FOUND IN {VOICE}')
    return found

# ----------------------------------------------------------- AUDIO CHAIN
def chain(ln=None):
    f = [f'highpass=f={HIGHPASS_HZ}']
    if DENOISE: f.append('afftdn=nr=12:nf=-30')
    f += [f'silenceremove=start_periods=1:start_duration=0:'
          f'start_threshold={TRIM_DB}dB:detection=peak',
          'areverse',
          f'silenceremove=start_periods=1:start_duration=0:'
          f'start_threshold={TRIM_DB}dB:detection=peak',
          'areverse']
    base = f'loudnorm=I={TARGET_I}:TP={TARGET_TP}:LRA={TARGET_LRA}'
    if ln is None:
        f.append(base + ':print_format=json')
    else:
        f.append(base + ':measured_I={input_i}:measured_LRA={input_lra}:'
                 'measured_TP={input_tp}:measured_thresh={input_thresh}:'
                 'offset={target_offset}:linear=true'.format(**ln))
        f += ['aformat=sample_fmts=s16:sample_rates=48000:channel_layouts=stereo',
              f'adelay={int(HEAD_PAD*1000)}:all=1',
              f'apad=pad_dur={TAIL_PAD}']
    return ','.join(f)

def measure(src):
    rc, out = sh(['ffmpeg', '-hide_banner', '-nostdin', '-i', str(src),
                  '-af', chain(None), '-f', 'null', '-'])
    blocks = re.findall(r'\{[^{}]*"input_i"[^{}]*\}', out, re.S)
    if not blocks: raise Fail(f'LOUDNORM MEASUREMENT FAILED FOR {src.name}\n{out[-1200:]}')
    d = json.loads(blocks[-1])
    for k in ('input_i', 'input_lra', 'input_tp', 'input_thresh', 'target_offset'):
        if k not in d or d[k] in ('-inf', 'inf', 'nan'):
            raise Fail(f'LOUDNORM RETURNED {k}={d.get(k)!r} FOR {src.name}; '
                       'THE TAKE IS PROBABLY SILENT')
    return d

def condition(src, dst, force):
    if dst.exists() and not force and dst.stat().st_mtime >= src.stat().st_mtime:
        return False
    ln = measure(src)
    rc, out = sh(['ffmpeg', '-hide_banner', '-nostdin', '-y', '-i', str(src),
                  '-af', chain(ln), '-ar', '48000', '-ac', '2',
                  '-c:a', 'pcm_s16le', str(dst)])
    if rc or not dst.exists(): raise Fail(f'CONDITIONING FAILED FOR {src.name}\n{out[-1200:]}')
    return True

def silence(dst, secs, force):
    if dst.exists() and not force: return False
    rc, out = sh(['ffmpeg', '-hide_banner', '-nostdin', '-y', '-f', 'lavfi',
                  '-i', f'anullsrc=r=48000:cl=stereo', '-t', f'{secs}',
                  '-c:a', 'pcm_s16le', str(dst)])
    if rc: raise Fail(f'COULD NOT MAKE SILENCE\n{out[-600:]}')
    return True

# ------------------------------------------------------------- METADATA
# docs/index.html READS chapters.json AS {total, chapters:[{start, title}, ...]}
# AND FALLS BACK TO ITS OWN BUILT IN MARKS IF THE FETCH FAILS, SO THE SHAPE
# BELOW IS A CONTRACT WITH THAT PAGE. THE EXTRA KEYS ARE IGNORED BY THE PAGE
# AND ARE THERE FOR READING BY A HUMAN.
def write_meta(plan, total):
    doc = {'total': round(total, 2),
           'slides': len(plan),
           'generator': 'BUILD_VIDEO.py, FROM RECORDED VOICE NOTES',
           'chapters': [{'start': round(it['start'], 2),
                         'end': round(it['start'] + it['dur'], 2),
                         'duration': round(it['dur'], 2),
                         'title': it['title'],
                         'slide': it['slide']['key'],
                         'kind': 'question' if it['src'] is None else 'answer'}
                        for it in plan]}
    (OUT.parent / 'chapters.json').write_text(json.dumps(doc, indent=1), encoding='utf-8')
    with (OUT.parent / 'timeline.csv').open('w', newline='', encoding='utf-8') as fh:
        w = csv.writer(fh)
        w.writerow(['slide', 'kind', 'start_s', 'duration_s', 'start_mmss', 'source', 'title'])
        for it in plan:
            w.writerow([it['slide']['key'],
                        'question' if it['src'] is None else 'answer',
                        f'{it["start"]:.2f}', f'{it["dur"]:.2f}', hms(it['start']),
                        it['src'].name if it['src'] else 'silence', it['title']])

# ------------------------------------------------------------------ MAIN
def main():
    force = '--force' in sys.argv
    check = '--check' in sys.argv
    meta  = '--meta'  in sys.argv
    for t in ('ffmpeg', 'ffprobe'): need(t)

    print('=' * 78)
    print('CML6032 ASSIGNMENT 1 - VIDEO FROM RECORDED VOICE NOTES')
    print('=' * 78)
    print(f'SLIDES  {SLIDES}\nVOICE   {VOICE}\nWORK    {WORK}\nOUT     {OUT}')
    print(f'DENOISE {"ON" if DENOISE else "OFF"}   QUESTION HOLD {QUESTION_HOLD:.1f} s   '
          f'CRF {CRF}   {FPS} fps\n')

    slides  = read_slides()
    answers = [s for s in slides if not s['suffix']]
    quests  = [s for s in slides if s['suffix']]
    notes   = read_notes()

    want = {s['n'] for s in answers}
    have = set(notes)
    if want - have:
        raise Fail('NO RECORDING FOR SLIDE(S) ' + ', '.join(str(n) for n in sorted(want - have)))
    if have - want:
        raise Fail('RECORDING(S) FOR SLIDE(S) ' + ', '.join(str(n) for n in sorted(have - want))
                   + ' BUT THERE IS NO SUCH ANSWER SLIDE')
    print(f'{len(answers)} ANSWER SLIDES, {len(quests)} QUESTION SLIDES, '
          f'{len(notes)} RECORDINGS, ALL PAIRED')

    raw = {n: duration(p) for n, p in sorted(notes.items())}
    med = sorted(raw.values())[len(raw) // 2]
    short = [n for n, d in raw.items() if d < SHORT_FACTOR * med]
    if short:
        msg = ('LIKELY TRUNCATED TAKE(S): ' +
               ', '.join(f'SLIDE {n} IS {raw[n]:.1f} s AGAINST A MEDIAN OF {med:.1f} s'
                         for n in sorted(short)))
        if STRICT: raise Fail(msg)
        print('\n*** WARNING *** ' + msg)
        print('    THE BUILD CONTINUES. RE-RECORD AND RERUN TO REPLACE IT.\n')

    WORK.mkdir(parents=True, exist_ok=True)
    norm = WORK / 'norm'; norm.mkdir(exist_ok=True)

    plan = []
    for s in slides:
        if s['suffix']:
            wav = norm / f'{s["key"]}.wav'
            plan.append({'slide': s, 'wav': wav, 'src': None,
                         'title': QUESTION_TITLES.get(s['key'], 'THE BRIEF AS ISSUED')})
        else:
            wav = norm / f'{s["n"]:02d}.wav'
            plan.append({'slide': s, 'wav': wav, 'src': notes[s['n']],
                         'title': TITLES.get(s['n'], f'SLIDE {s["n"]}')})

    if check:
        print('PLAN (RAW RECORDING LENGTHS, BEFORE TRIM AND PAD)')
        for it in plan:
            s = it['slide']
            d = raw[s['n']] if it['src'] else QUESTION_HOLD
            tag = 'HOLD ' if it['src'] is None else 'VOICE'
            print(f'  {s["key"]:>4}  {tag}  {d:7.2f} s  '
                  f'{it["src"].name if it["src"] else "(silence)"}')
        print(f'\nRAW TOTAL {sum(raw.values()) + QUESTION_HOLD*len(quests):.1f} s')
        return

    print('CONDITIONING AUDIO')
    for it in plan:
        if it['src'] is None:
            did = silence(it['wav'], QUESTION_HOLD, force)
        else:
            did = condition(it['src'], it['wav'], force)
        it['dur'] = duration(it['wav'])
        print(f'  {it["slide"]["key"]:>4}  {it["dur"]:7.2f} s  '
              f'{"BUILT" if did else "CACHED"}  '
              f'{it["src"].name if it["src"] else "silence"}')

    t = 0.0
    for it in plan:
        it['start'] = t; t += it['dur']
    total = t
    print(f'\nTIMELINE {total:.2f} s = {hms(total)}')

    if meta:
        write_meta(plan, total)
        print('REWROTE chapters.json AND timeline.csv FROM THE CACHED AUDIO, NO ENCODE')
        for it in plan:
            print(f'  {hms(it["start"])}  {it["slide"]["key"]:>4}  {it["dur"]:6.2f} s  {it["title"]}')
        return

    alist = WORK / 'audio.txt'
    alist.write_text(''.join(f"file '{it['wav'].as_posix()}'\n" for it in plan), encoding='utf-8')
    audio = WORK / 'audio.wav'
    rc, out = sh(['ffmpeg', '-hide_banner', '-nostdin', '-y', '-f', 'concat', '-safe', '0',
                  '-i', str(alist), '-c:a', 'pcm_s16le', '-ar', '48000', '-ac', '2', str(audio)])
    if rc: raise Fail('AUDIO CONCATENATION FAILED\n' + out[-1200:])
    ad = duration(audio)
    if abs(ad - total) > 0.25:
        raise Fail(f'CONCATENATED AUDIO IS {ad:.2f} s BUT THE TIMELINE SAYS {total:.2f} s')

    ilist = WORK / 'images.txt'
    lines = []
    for it in plan:
        lines.append(f"file '{it['slide']['path'].as_posix()}'\nduration {it['dur']:.6f}\n")
    lines.append(f"file '{plan[-1]['slide']['path'].as_posix()}'\n")   # CONCAT DEMUXER EATS THE LAST DURATION
    ilist.write_text(''.join(lines), encoding='utf-8')

    venc = ['-c:v', 'libx264', '-preset', 'medium', '-crf', str(CRF), '-tune', 'stillimage']
    rc, _ = sh(['ffmpeg', '-hide_banner', '-encoders'])
    if 'h264_nvenc' in _ and os.environ.get('CML_NVENC', '1') == '1':
        venc = ['-c:v', 'h264_nvenc', '-preset', 'p5', '-rc', 'vbr', '-cq', str(CRF), '-b:v', '0']
        print('USING h264_nvenc')
    else:
        print('USING libx264')

    print('ENCODING, ONE PASS')
    cmd = ['ffmpeg', '-hide_banner', '-nostdin', '-y',
           '-f', 'concat', '-safe', '0', '-i', str(ilist),
           '-i', str(audio),
           '-vf', f'scale=1920:1080:force_original_aspect_ratio=decrease,'
                  f'pad=1920:1080:(ow-iw)/2:(oh-ih)/2:color=black,fps={FPS},format=yuv420p',
           *venc, '-c:a', 'aac', '-b:a', ABR, '-ar', '48000', '-ac', '2',
           '-movflags', '+faststart', '-shortest', str(OUT)]
    rc, out = sh(cmd)
    if rc or not OUT.exists(): raise Fail('ENCODE FAILED\n' + out[-2500:])

    got = duration(OUT)
    if abs(got - total) > 0.5:
        raise Fail(f'OUTPUT IS {got:.2f} s BUT THE TIMELINE SAYS {total:.2f} s')

    write_meta(plan, total)

    print('\n' + '=' * 78)
    print(f'WROTE {OUT}')
    print(f'      {got:.2f} s = {hms(got)}   {OUT.stat().st_size/1e6:.1f} MB   1920x1080 @ {FPS} fps')
    print(f'      chapters.json AND timeline.csv BESIDE IT')
    print('=' * 78)
    for it in plan:
        print(f'  {hms(it["start"])}  {it["slide"]["key"]:>4}  {it["dur"]:6.2f} s  {it["title"]}')

if __name__ == '__main__':
    main()



## 4. SETTINGS

EVERY KNOB IS AN ENVIRONMENT VARIABLE, SO NOTHING IN THE SCRIPT NEEDS EDITING.

| VARIABLE | DEFAULT | MEANING |
|---|---|---|
| `CML_SLIDES` | `./slides` | SLIDE PNG DIRECTORY |
| `CML_VOICE` | `./voicenotes` | RECORDINGS DIRECTORY |
| `CML_WORK` | `./build` | SCRATCH AND PER SLIDE WAV CACHE |
| `CML_OUT` | `./CML6032_Assignment1_Nisheal_2025CYS7090.mp4` | OUTPUT |
| `CML_QUESTION_HOLD` | `8.0` | SECONDS EACH QUESTION SLIDE IS HELD SILENT |
| `CML_HEAD_PAD` / `CML_TAIL_PAD` | `0.25` / `0.60` | LEAD IN AND LEAD OUT PER SLIDE |
| `CML_DENOISE` | `0` | `1` ENABLES A GENTLE FFT DENOISE |
| `CML_CRF` | `20` | H.264 QUALITY, LOWER IS BIGGER AND BETTER |
| `CML_ABR` | `160k` | AAC BITRATE |
| `CML_FPS` | `30` | FRAME RATE |
| `CML_STRICT` | `0` | `1` MAKES A SUSPICIOUSLY SHORT TAKE FATAL |


In [ ]:
os.environ['CML_SLIDES'] = 'slides'
os.environ['CML_VOICE']  = 'voicenotes'
os.environ['CML_WORK']   = 'build'
os.environ['CML_OUT']    = 'CML6032_Assignment1_Nisheal_2025CYS7090.mp4'
os.environ['CML_QUESTION_HOLD'] = '8.0'
os.environ['CML_DENOISE'] = '0'
os.environ['CML_CRF']     = '20'
for k, v in sorted(os.environ.items()):
    if k.startswith('CML_'): print(f'{k:22} {v}')


## 5. DRY RUN FIRST

THIS VALIDATES THE PAIRING AND PRINTS THE PLAN WITHOUT TOUCHING FFMPEG. RUN IT
BEFORE THE BUILD EVERY TIME. IT IS FATAL IF A SLIDE HAS NO RECORDING, IF A
RECORDING MATCHES NO SLIDE, OR IF TWO RECORDINGS CLAIM THE SAME SLIDE, AND IT
WARNS IF ANY TAKE IS SHORTER THAN 0.35 OF THE MEDIAN, WHICH IS THE SIGNATURE OF
A RECORDING THAT STOPPED EARLY.


In [ ]:
!python3 BUILD_VIDEO.py --check


## 6. BUILD

CONDITIONED PER SLIDE WAV FILES ARE CACHED IN `build/norm/`, SO A RERUN AFTER
REPLACING ONE TAKE ONLY RECONDITIONS THAT TAKE. `--force` RECONDITIONS ALL.


In [ ]:
!python3 BUILD_VIDEO.py


## 7. VERIFY

CHECK THE CONTAINER, THEN THE PER SLIDE LOUDNESS, THEN THE CHAPTER MARKS. THE
SCRIPT ALREADY ASSERTS THAT THE FINISHED DURATION MATCHES THE PLANNED TIMELINE
TO WITHIN 0.5 s, SO THESE CELLS ARE FOR YOUR EYES, NOT FOR CORRECTNESS.


In [ ]:
import json, subprocess
OUT = os.environ['CML_OUT']
print(subprocess.run(['ffprobe','-v','error','-show_entries',
  'format=duration,size,bit_rate:stream=codec_name,width,height,r_frame_rate,sample_rate,channels',
  '-of','default=nw=1', OUT], capture_output=True, text=True).stdout)
print(open('timeline.csv').read())


In [ ]:
# PER SLIDE INTEGRATED LOUDNESS AFTER CONDITIONING. EVERY VALUE SHOULD SIT
# WITHIN ABOUT 1 LU OF -16 LUFS; A LARGE OUTLIER MEANS THAT TAKE IS ODD.
import glob, re, subprocess
for w in sorted(glob.glob('build/norm/*.wav')):
    r = subprocess.run(['ffmpeg','-hide_banner','-nostdin','-i',w,'-af',
                        'ebur128=framelog=quiet','-f','null','-'],
                       capture_output=True, text=True).stderr
    m = re.findall(r'I:\s*(-?\d+\.\d+) LUFS', r)
    print(f"{os.path.basename(w):>8}  {m[-1] if m else 'n/a':>8} LUFS")


In [ ]:
import json
ch = json.load(open('chapters.json'))
for c in ch:
    print(f"{int(c['t'])//60:02d}:{int(c['t'])%60:02d}  {c['slide']:>4}  {c['kind']:<8} {c['title']}")


## 8. PREVIEW

A FULL 22 MINUTE FILE IS TOO BIG TO EMBED IN A NOTEBOOK OUTPUT, SO THIS CUTS A
SHORT EXCERPT AROUND A CHAPTER OF YOUR CHOICE AND PLAYS THAT. SET `AT` TO ANY
START TIME FROM THE CHAPTER LIST ABOVE.


In [ ]:
from IPython.display import HTML
import base64, subprocess
AT, LEN = 0, 45     # SECONDS: START, LENGTH
subprocess.run(['ffmpeg','-hide_banner','-nostdin','-y','-ss',str(AT),'-t',str(LEN),
                '-i',os.environ['CML_OUT'],'-vf','scale=960:-2','-c:v','libx264',
                '-crf','28','-c:a','aac','-b:a','96k','preview.mp4'], check=True)
b = base64.b64encode(open('preview.mp4','rb').read()).decode()
HTML(f'<video width=720 controls src="data:video/mp4;base64,{b}"></video>')


## 9. COLLECT THE OUTPUT

IF YOU USED DRIVE, THE MP4 IS ALREADY THERE. OTHERWISE DOWNLOAD IT. A 22 MINUTE
1080p SLIDE VIDEO LANDS AROUND 60 TO 90 MB, WHICH IS UNDER THE 100 MB GITHUB
BLOB LIMIT, BUT CHECK BEFORE COMMITTING; IF IT IS OVER, ATTACH IT TO A GITHUB
RELEASE INSTEAD AND POINT `docs/index.html` AT THE RELEASE ASSET. DO NOT USE GIT
LFS ON GITHUB PAGES, BECAUSE PAGES SERVES THE LFS POINTER TEXT AND NOT THE FILE.


In [ ]:
print(f"{os.path.getsize(os.environ['CML_OUT'])/1e6:.1f} MB")
# from google.colab import files
# files.download(os.environ['CML_OUT'])
# files.download('chapters.json')


## 10. PUTTING IT ON THE SITE

`docs/index.html` IN THE REPOSITORY IS ALREADY A SELF HOSTED PLAYER. IT LOOKS
FOR THE MP4 AND FOR `chapters.json` BESIDE ITSELF, AND THE CHAPTER STRIP IS
BUILT FROM WHATEVER `chapters.json` CONTAINS, SO THE THREE QUESTION SLIDES SHOW
UP AS THEIR OWN MARKS WITH NO EDIT TO THE PAGE.

1. COPY THE MP4 AND `chapters.json` INTO `docs/`
2. CHECK THE FILENAME IN `docs/index.html` MATCHES THE MP4
3. COMMIT, THEN SETTINGS > PAGES > BRANCH `main`, FOLDER `/docs`
4. OPEN THE LIVE URL IN A LOGGED OUT BROWSER AND CONFIRM IT PLAYS

**NOT YET.** NOTHING IN THE REPOSITORY HAS BEEN TOUCHED FOR THIS CHANGE. DO THIS
ONLY AFTER YOU HAVE WATCHED THE VIDEO AND ARE SATISFIED WITH IT.
